In [53]:
CREATE_RESUME_PROMPT = """
Create one professional, ATS-friendly resume in Markdown using the
USER DATA, EXISTING RESUME, and JOB DATA provided below.

The EXISTING RESUME is the source of truth for the candidate's
professional background. It may contain information from one or
multiple original resumes.

The JOB DATA is only used to understand the target role and prioritize
relevant information from the EXISTING RESUME.

USER DATA is used for identity and contact information.

RULES:

- Never invent or assume candidate information.
- Never add a skill, technology, responsibility, qualification,
  achievement, metric, company, title, date, project, certification,
  education, or experience merely because it appears in the JOB DATA.
- Never calculate or guess years of experience.
- Never convert internship or project experience into professional
  experience.
- Preserve the factual meaning of the EXISTING RESUME.
- You may rewrite, shorten, reorder, and improve existing content.
- Prioritize experience, projects, and skills relevant to the JOB DATA.
- Use only skills supported by the EXISTING RESUME.
- Create the professional summary using only facts from the EXISTING
  RESUME.
- Avoid generic filler such as "highly motivated", "detail-oriented",
  "passionate", "results-driven", or "strong foundation".

If the EXISTING RESUME contains information from multiple resumes,
merge duplicate or overlapping information and produce ONE coherent
resume. Do not simply concatenate the content.

CONTACT INFORMATION:

Use USER DATA for name, email, phone, LinkedIn, GitHub, and portfolio.
If a value is missing from USER DATA but exists in the EXISTING RESUME,
you may use it.

Never guess or generate contact information.

When an actual email address is available, render it as a Markdown
mailto link:

[dasaniran268@gmail.com](mailto:dasaniran268@gmail.com)

When an actual LinkedIn, GitHub, or portfolio URL is available, render
it as a Markdown hyperlink:

[LinkedIn](ACTUAL_URL)
[GitHub](ACTUAL_URL)
[Portfolio](ACTUAL_URL)

Do not invent URLs.

If the source only contains a label such as "LinkedIn" or "GitHub"
without an actual URL, keep the label as plain text.

RESUME:

Create a conventional 1–2 page professional resume using sections
such as:

# Name

Contact Information

## Professional Summary

## Technical Skills

## Professional Experience

## Projects

## Education

## Achievements

## Certifications

## Competitive Programming

Only include sections for which information exists.

Use clean Markdown with headings, bullets, bold text, and Markdown
hyperlinks.

Do not use HTML, CSS, tables, emojis, icons, or code fences.

OUTPUT:

The `markdown` field must contain ONLY the actual resume.

The first content must be the candidate's name.

Do not include explanations, reasoning, notes, comments, introductions,
conclusions, or any text outside the resume.

USER DATA:
{user_object}

EXISTING RESUME:
{existing_resume}

JOB DATA:
{job_object}

ADDITIONAL USER INSTRUCTION:
{user_instruction}

{format_instructions}
"""

In [54]:
from pydantic import BaseModel, Field


class CreateResumeResponse(BaseModel):
    resume: str = Field(
        description=(
           "ONLY the complete resume in Markdown format. "
            "Must begin with the candidate's name and contain no "
            "explanation, commentary, metadata, introduction, conclusion, "
            "PDF-generation statement, ATS statement, or any text that "
            "is not part of the actual resume."
        )
    )

In [55]:
import os
import json

from dotenv import load_dotenv
from pydantic import ValidationError

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

# from .create_resume_prompt import CREATE_RESUME_PROMPT
# from .create_resume_schema import CreateResumeResponse


load_dotenv()


llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    max_new_tokens=2500,
    temperature=0.2,
)

model = ChatHuggingFace(llm=llm)


parser = PydanticOutputParser(
    pydantic_object=CreateResumeResponse
)


fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


prompt = ChatPromptTemplate.from_template(
    CREATE_RESUME_PROMPT
)


create_resume_chain = prompt | model | fixing_parser


def create_resume(
    user: dict,
    resume: str,
    job: dict,
    user_instruction: str = "",
) -> CreateResumeResponse:

    if not user:
        raise ValueError("user must not be empty")

    if not resume or not resume.strip():
        raise ValueError("resume must not be empty")

    if not job:
        raise ValueError("job must not be empty")

    safe_user = {
        "full_name": user.get("full_name", ""),
        "email": user.get("email", ""),
        "phone": user.get("phone", ""),
        "linkedin_url": user.get("linkedin_url", ""),
        "github_url": user.get("github_url", ""),
        "portfolio_url": user.get("portfolio_url", ""),
    }

    safe_job = {
        "title": job.get("title", ""),
        "company": job.get("company", ""),
        "cities": job.get("cities", []),
        "countries": job.get("countries", []),
        "is_remote": job.get("is_remote", False),
        "is_hybride": job.get("is_hybride", False),
        "is_onsite": job.get("is_onsite", False),
        "required_skills": job.get("required_skills", []),
        "description": job.get("description", ""),
    }

    result = create_resume_chain.invoke({
        "user_object": json.dumps(
            safe_user,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),

        "existing_resume": resume,

        "job_object": json.dumps(
            safe_job,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),

        "user_instruction": (
            user_instruction.strip()
            if user_instruction
            else "No additional instruction."
        ),

        "format_instructions": parser.get_format_instructions(),
    })

    return result

In [56]:
user = {
    "full_name": "Anirban Das",
    "email": "dasaniran268@gmail.com",
    "phone": "+91 629035587",
    "linkedin_url": "https://linkedin.com/in/anirban-das",
    "github_url": "https://github.com/anirban-das",
    "portfolio_url": "https://anirbandas.dev",
}

In [57]:
resume = """

=====================
RESUME 1
=====================
# **Anirban Das** 

(+91) 629035587 _|_ dasaniran268@gmail.com _|_ LinkedIn _|_ GitHub 

## **Education** 

### **Indian Institute of Technology (ISM), Dhanbad, India** 

Bachelor of Technology (CGPA: 8.29/10) 2023 – **Relevant Coursework:** Data Structures & Algorithms, Object Oriented Programming (C++), Operating Systems, Database Management Systems (DBMS), Compiler Design 

2023 – 2027 

## **Experience** 

### **Google** _|_ **Software Engineer Intern** 

   - May 2026 – Jul 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## **Projects** 

**Code Fusion** _|_ **React.js, MongoDB, Yjs, Socket.IO, ExpressJS** 

Deployed _|_ GitHub 

- Built a CRDT-based real-time collaborative code editor using Yjs and Monaco Editor, enabling conflict-free multi-user editing through delta-based synchronization and awareness protocol for live cursor tracking across sessions. 

- Developed a real-time collaboration system supporting live cursor tracking and integrated in-app chat, allowing seamless communication between distributed developers during shared coding sessions. 

- Implemented multi-language support, auto-completion, real-time syntax error detection, persistent code storage, customizable themes, and a collapsible file explorer for efficient workflow management. 

### **JobPilot** _|_ **FastAPI, Next.js, Socket.IO, LangChain, Tailwind CSS** 

   - GitHub 

- Developed an automation system using FastAPI, Next.js, and HuggingFace LLMs to automate resume extraction and job matching, reducing manual application time by 80%. 

- Architected a scalable asynchronous worker system with WebSockets/Socket.IO updates and FileLock synchronization to manage concurrent multi-user job lifecycles. 

- Built an LLM-powered job ranking and categorization pipeline using prompt engineering, along with a manual review workflow and local validation setup. 

## **Competitive Programming** 

**LeetCode:** aswU2SZvDg _|_ Solved 240+ problems focusing on data structures and algorithms **CodeForces:** anirban2005 _|_ Max Rating: 1216 (Pupil) _|_ Solved 390+ problems 

## **Technical Skills** 

- **Languages & Databases:** C++, Python, JavaScript, TypeScript, MongoDB, PostgreSQL, Vector Databases 

- **Core Frameworks:** React.js, Next.js, Node.js, Express.js, FastAPI, Tailwind CSS, Three.js 

- **AI/ML & Tools:** LangChain, LangGraph, TensorFlow, Keras, Git, Docker, Postman 

## **Achievements** 

- Secured **4th** rank at **HaXplore** | **CodeFest’25** , organized by **IIT BHU** . 

- **Winner** of Winter of Code 6.0 (Web Development Division), a one-month long hackathon conducted by **CyberLabs** , IIT (ISM) Dhanbad. | **Deployed Project** 



=====================
RESUME 2
=====================
# **Anirban Das** 

**Adm. No.** 23JE0104 � 6290375587 � My <u>portfolio website</u> � dasanirban268@gmail.com � <u>linkedin.com/in/anirbandas</u> � <u>github.com/anirban2005143a</u> 



## Education 

### **Indian Institute of Technology (Indian School of Mines), Dhanbad** 

_Bachelor of Technology in Computer Science and Engineering (GPA: 8.29 / 10.00)_ 

Expected May 2027 _Dhanbad, Jharkhand_ 

- **Relevant Coursework:** Data Structures and Algorithms (C++), Database Management System, Compiler Design, Computer Organization , Computer Architecture, Operating Systems. 

## Experience 

### **Google** _|_ **_Software Engineer Intern_** 

May 2026 – July 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## Projects 

**<u>Code Fusion</u>** _| React.js, Flask, Express.js, MongoDB, Tailwind CSS |_ _<u>GitHub</u> | Deployed Project_ 

- An online code editor supporting real-time collaboration, multiple languages, and customizable themes. 

- Enables multiple developers to collaborate in real-time with live cursor tracking and integrated in-app chat for seamless communication. 

- Integrates auto-completion, real-time syntax error detection, and persistent code-saving functionality, while offering various themes, multi-language support, and a collapsible sidebar for efficient file management. 

**<u>JobPilot</u>** _| Python (FastAPI), TypeScript (Next.js), Langchain, WebSockets |_ _<u>GitHub</u> |_ _<u>Video</u>_ 

- Developed a full-stack automation system using **FastAPI** , **Next.js** , and **HuggingFace LLMs** to automate resume extraction and job matching, reducing manual application time by **80%** . 

- Architected a scalable background worker system with **asynchronous processing** , **WebSockets** for real-time updates, and **FileLock** synchronization to manage concurrent multi-user job lifecycles. 

- Built an **LLM-powered** pipeline using **prompt engineering** for job ranking and categorization, featuring a manual review workflow and a local mock portal for safe system validation. 

**NoteBridge** _| React.js, Express.js, MongoDB, Bootstrap |_ _<u>GitHub</u> | Deployed Project_ 

- A **feature-rich note-taking and sharing platform** that enables **structured organization** through folders and facilitates **controlled file sharing** . 

- Enables **interactive engagement** through features like **likes** , **comments** , and **shares** . 

- Provides a **comprehensive profile page** displaying total posts, followers, following, and an **organized archive of past posts** for easy access and engagement. 

## Technical Skills 

**AI/ML & Agents** : LangChain, LangGraph, TensorFlow, Keras, Deep Learning, ANN, CNN, LSTM. **Technologies** : Node.js, FastAPI, Express.js, Docker, Next.js, React.js, Tailwind CSS, Three.js, GSAP. **Database & Cloud** : MongoDB, PostgreSQL, Vector Databases. 

## Achievements 

- Secured **4th** rank at **HaXplore** _|_ **CodeFest’25** , organized by **IIT BHU!** 

- **Winner** - of Winter Of Code 6.O (in Web Development Division) a one-month long hackathon conducted by **CyberLabs** , IIT(ISM) Dhanbad. _| Deployed Project_ 

## Social Engagements 

- Member of CyberLabs -Tech society of IIT ISM Dhanbad 

- Member of Aquatics Team - Swimming Team of IIT ISM Dhanbad. 

- Represented IIT Dhanbad at the 37th INTER IIT AQUATICS MEET 2023 held at IIT Gandhinagar and secured **4th place** in 200m Individual Medley . 
"""

In [58]:
job = {
    "_id": "6a835b392645704f6b223099",
    "sourceId": "6a835b392645704f6b223098",
    "jobId": "job-168",

    "cities": [
        "Hyderabad",
        "Pune"
    ],

    "company": "FinStack Labs",

    "countries": [
        "India"
    ],

    "createdAt": "2026-08-18T05:30:00.000Z",

    "description": """
We are looking for a Full Stack Software Engineer to build and maintain
high-quality financial technology products used by thousands of customers.

The engineer will work across frontend and backend systems and collaborate
closely with product managers, designers, QA engineers, and infrastructure
teams.

Responsibilities:

- Design and develop responsive web applications using React and Next.js.
- Build backend APIs and services using Node.js and TypeScript.
- Develop reusable frontend components and maintain a scalable frontend
  architecture.
- Design and optimize MongoDB data models and database queries.
- Integrate third-party APIs and external services.
- Implement authentication, authorization, and secure API endpoints.
- Write unit, integration, and end-to-end tests.
- Containerize applications using Docker and support CI/CD workflows.
- Monitor application performance and troubleshoot production issues.
- Participate in code reviews and contribute to engineering standards.
- Work with product and design teams to deliver customer-facing features.

Requirements:

- 2+ years of professional software engineering experience.
- Strong proficiency in TypeScript and JavaScript.
- Strong experience with React and Next.js.
- Experience building backend services with Node.js.
- Solid understanding of MongoDB and database design.
- Experience developing and consuming REST APIs.
- Experience with Docker and containerized applications.
- Familiarity with AWS services such as ECS, Lambda, S3, and CloudWatch.
- Experience with automated testing using Jest, Playwright, or Cypress.
- Understanding of authentication mechanisms such as OAuth2 and JWT.
- Familiarity with CI/CD pipelines and GitHub Actions.
- Strong understanding of web application security and performance
  optimization.

Nice to have:

- Experience with PostgreSQL.
- Experience with Redis.
- Experience working with WebSockets.
- Experience with real-time collaborative applications.
- Experience working in a fast-paced startup environment.
- Experience with AI-powered product features.
    """,

    "is_hybride": True,
    "is_onsite": False,
    "is_remote": True,

    "required_skills": [
        "TypeScript",
        "JavaScript",
        "React",
        "Next.js",
        "Node.js",
        "MongoDB",
        "REST APIs",
        "Docker",
        "AWS",
        "ECS",
        "Lambda",
        "S3",
        "CloudWatch",
        "Jest",
        "Playwright",
        "Cypress",
        "OAuth2",
        "JWT",
        "GitHub Actions",
        "CI/CD",
        "Web Security",
        "Performance Optimization"
    ],

    "salary_offered": "₹24,00,000 - ₹36,00,000 per year",

    "start_date": "2026-11-01",

    "status": "discovered",

    "title": "Full Stack Software Engineer",

    "updatedAt": "2026-08-18T05:30:00.000Z",

    "visa_sponsorship_offered": False
}

In [ ]:
result = create_resume(
    user=user,
    resume=resume,
    job=job,
    user_instruction=""
)

print(result.resume)